<a href="https://colab.research.google.com/github/DavidRuiz-beep/Proyecto/blob/main/2_Taller_Regresion_Polinomica_David_Ruiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/5_Aprendizaje_supervisado/2_Taller_Regresion_Polinomica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# **Taller: Regresión Polinómica, Subajuste, Sobreajuste y Regularización**

**IMPORTANTE**: Guarda una copia de este notebook en tu Google Drive o computador.

**Taller en grupos de 3**

**Nombres estudiantes:**

- David Felipe Ruiz Parra

**Forma de entrega:**

- Nombrar el archivo de la siguiente forma: “Taller_Reg_Pol_apellidos.ipynb”.
- Suba el Jupyter Notebook a su cuenta en Github y envíe el link en el siguiente Forms: https://forms.cloud.microsoft/r/AUmhqMjhUK.

**IMPORTANTE:** No se recibirán talleres en Google Colab, el notebook debe estar subido en Github.

**Plazo de entrega:**

30 de abril de 2026, máximo a las 11:59 p.m. Tenga en cuenta que luego de esa hora el formulario en forms se cierra. El Jupupyter Notebook también debe quedar subido en Github antes de esa hora.

**Instrucciones Generales:**

Completa el código en las celdas marcadas con `### TU CÓDIGO AQUÍ ###`. Puedes añadir más celdas si lo requieres.

## **Situación**

Una importante firma de inversión inmobiliaria te ha contratado como consultor de ciencia de datos. Su proceso actual de valoración de propiedades es lento y se basa en la intuición de unos pocos expertos. Quieren que desarrolles un modelo de machine learning para predecir el precio de venta de las viviendas (`SalePrice`) de forma más precisa y sistemática.

Te han entregado el dataset "Ames Housing", que contiene una gran cantidad de información sobre viviendas vendidas recientemente. Tu tarea es construir el mejor modelo posible, pero más importante aún, justificar por qué tu modelo es robusto y fiable, explicando cómo has manejado la complejidad y el riesgo de sobreajuste.

### **Ejercicio 1: Carga y Preparación Inicial de los Datos**

Primero, carguemos las librerías necesarias y el dataset. Nos enfocaremos en un subconjunto de las variables numéricas para mantener el taller manejable, pero el principio se aplica a todo el dataset.

In [1]:
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

Mejorar visualización de dataframes y gráficos

In [2]:
# Que muestre todas las columnas
pd.options.display.max_columns = None
# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.2f}'.format

# Configuraciones para una mejor visualización
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

Cargar el dataset

In [3]:
# Usamos el dataset Ames Housing desde su fuente original (requiere internet)
url = 'http://jse.amstat.org/v19n3/decock/AmesHousing.txt'
df = pd.read_csv(url, sep='\t')
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,Condition 2,Bldg Type,House Style,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Roof Style,Roof Matl,Exterior 1st,Exterior 2nd,Mas Vnr Type,Mas Vnr Area,Exter Qual,Exter Cond,Foundation,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin SF 1,BsmtFin Type 2,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Heating,Heating QC,Central Air,Electrical,1st Flr SF,2nd Flr SF,Low Qual Fin SF,Gr Liv Area,Bsmt Full Bath,Bsmt Half Bath,Full Bath,Half Bath,Bedroom AbvGr,Kitchen AbvGr,Kitchen Qual,TotRms AbvGrd,Functional,Fireplaces,Fireplace Qu,Garage Type,Garage Yr Blt,Garage Finish,Garage Cars,Garage Area,Garage Qual,Garage Cond,Paved Drive,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.00,31770,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,5,1960,1960,Hip,CompShg,BrkFace,Plywood,Stone,112.00,TA,TA,CBlock,TA,Gd,Gd,BLQ,639.00,Unf,0.00,441.00,"1,080.00",GasA,Fa,Y,SBrkr,1656,0,0,1656,1.00,0.00,1,0,3,1,TA,7,Typ,2,Gd,Attchd,"1,960.00",Fin,2.00,528.00,TA,TA,P,210,62,0,0,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.00,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,Norm,1Fam,1Story,5,6,1961,1961,Gable,CompShg,VinylSd,VinylSd,NaN,0.00,TA,TA,CBlock,TA,TA,No,Rec,468.00,LwQ,144.00,270.00,882.00,GasA,TA,Y,SBrkr,896,0,0,896,0.00,0.00,1,0,2,1,TA,5,Typ,0,NaN,Attchd,"1,961.00",Unf,1.00,730.00,TA,TA,Y,140,0,0,0,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.00,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,6,1958,1958,Hip,CompShg,Wd Sdng,Wd Sdng,BrkFace,108.00,TA,TA,CBlock,TA,TA,No,ALQ,923.00,Unf,0.00,406.00,"1,329.00",GasA,TA,Y,SBrkr,1329,0,0,1329,0.00,0.00,1,1,3,1,Gd,6,Typ,0,NaN,Attchd,"1,958.00",Unf,1.00,312.00,TA,TA,Y,393,36,0,0,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.00,11160,Pave,NaN,Reg,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,7,5,1968,1968,Hip,CompShg,BrkFace,BrkFace,NaN,0.00,Gd,TA,CBlock,TA,TA,No,ALQ,"1,065.00",Unf,0.00,"1,045.00","2,110.00",GasA,Ex,Y,SBrkr,2110,0,0,2110,1.00,0.00,2,1,3,1,Ex,8,Typ,2,TA,Attchd,"1,968.00",Fin,2.00,522.00,TA,TA,Y,0,0,0,0,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.00,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,5,5,1997,1998,Gable,CompShg,VinylSd,VinylSd,NaN,0.00,TA,TA,PConc,Gd,TA,No,GLQ,791.00,Unf,0.00,137.00,928.00,GasA,Gd,Y,SBrkr,928,701,0,1629,0.00,0.00,2,1,3,1,TA,6,Typ,1,TA,Attchd,"1,997.00",Fin,2.00,482.00,TA,TA,Y,212,34,0,0,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [4]:
# Para este taller, solo usaremos algunas columnas clave para simplificar.
df_sub = df[['Overall Qual', 'Gr Liv Area', 'SalePrice']].copy()
df_sub.columns = ['OverallQual', 'GrLivArea', 'SalePrice'] # Renombrar para facilidad
df_sub.head()

,OverallQual,GrLivArea,SalePrice
0,6,1656,215000
1,5,896,105000
2,6,1329,172000
3,7,2110,244000
4,5,1629,189900


**Explicación de las variables del dataset reducido**

1. SalePrice: Precio de Venta.

Esta es la variable objetivo. Es el precio final por el cual se vendió la propiedad, medido en dólares estadounidenses.

2. OverallQual: Calidad General.

Es una variable ordinal que califica la calidad general del material y el acabado de la casa. Es una de las variables predictoras (X) más importantes.

Escala: Va de 1 a 10.

10: Muy Excelente

9: Excelente

...

2: Pobre

1: Muy Pobre

3. GrLivArea: Área Habitable sobre el Nivel del Suelo.

Es una variable numérica que mide el total de metros cuadrados de área habitable que está por encima del nivel del suelo. No incluye el área del sótano. Es una de las variables predictoras (X) más fuertes, ya que, lógicamente, casas más grandes tienden a ser más caras.

In [5]:
df_sub.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   OverallQual  2930 non-null   int64
 1   GrLivArea    2930 non-null   int64
 2   SalePrice    2930 non-null   int64
dtypes: int64(3)
memory usage: 68.8 KB


### **Ejercicio 2: Dividir el conjunto de datos**

El método `.info()` muestra que no hay nulos en nuestro subconjunto. ¡Perfecto! Ahora, definamos nuestras variables `X` (predictoras) e `y` (objetivo) y dividamos los datos.

In [6]:
# Definir X e y
X = df_sub[['OverallQual', 'GrLivArea']]
y = df_sub['SalePrice']

In [7]:
# Dividir en entrenamiento y prueba (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

### **Ejercicio 3: Modelo con Sobreajuste**

Vamos a crear un modelo muy complejo para ver qué tan mal puede generalizar.

**Crear un Modelo Polinómico de Grado 5**

Usa `Pipeline` para combinar `PolynomialFeatures` (grado 5), `StandardScaler` y `LinearRegression`. Entrénalo con los datos de entrenamiento.

In [8]:
# Crear el pipeline para el modelo polinómico
poli_pipeline = Pipeline([
    ('poly_features', PolynomialFeatures(degree=5, include_bias=False)),
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

In [9]:
# Entrenar el modelo
poli_pipeline.fit(X_train, y_train)

Pipeline(steps=[('poly_features',
                 PolynomialFeatures(degree=5, include_bias=False)),
                ('scaler', StandardScaler()),
                ('regressor', LinearRegression())])

In [10]:
# Calcular el error (RMSE) en entrenamiento y prueba y realizar un print de estos
y_train_pred = poli_pipeline.predict(X_train)
y_test_pred = poli_pipeline.predict(X_test)

rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f"Modelo Polinómico grado 5:")
print(f"  RMSE entrenamiento: {rmse_train:,.0f}")
print(f"  RMSE prueba:        {rmse_test:,.0f}")


Modelo Polinómico grado 5:
  RMSE entrenamiento: 34,152
  RMSE prueba:        41,102


**Pregunta:** ¿Qué indica la diferencia que observas entre el error de entrenamiento y el de prueba? ¿Le recomendarías este modelo a la firma inmobiliaria? ¿Por qué?

La gran diferencia entre el error de entrenamiento (muy bajo) y el de prueba (alto) indica sobreajuste, el modelo ha "memorizado" los datos de entrenamiento, incluyendo ruido y patrones demasiado específicos, por lo que generaliza mal a datos nuevos, no recomendaría este modelo porque no es fiable para predecir precios de viviendas no vistas.


### **Ejercicio 4: Aplicar Regularización**

Ahora, vamos a "curar" el sobreajuste. Usaremos los mismos `PolynomialFeatures` de grado 5, pero cambiaremos el modelo de regresión.

**Implementar Regresión Ridge**

Copia el pipeline anterior, pero reemplaza `LinearRegression` con `Ridge(alpha=10)`. `alpha` es la fuerza de la penalización.

In [11]:
# Crear el pipeline para Ridge
ridge_pipeline = Pipeline([
    ('poly_features', PolynomialFeatures(degree=5, include_bias=False)),
    ('scaler', StandardScaler()),
    ('regressor', Ridge(alpha=10))
])

In [12]:
# Entrenar el modelo
ridge_pipeline.fit(X_train, y_train)

y_train_pred_ridge = ridge_pipeline.predict(X_train)
y_test_pred_ridge = ridge_pipeline.predict(X_test)

rmse_train_ridge = np.sqrt(mean_squared_error(y_train, y_train_pred_ridge))
rmse_test_ridge = np.sqrt(mean_squared_error(y_test, y_test_pred_ridge))

In [13]:
# Calcular el error (RMSE) en entrenamiento y prueba y realizar un print de estos
print(f"Modelo Ridge (alpha=10):")
print(f"  RMSE entrenamiento: {rmse_train_ridge:,.0f}")
print(f"  RMSE prueba:        {rmse_test_ridge:,.0f}")


Modelo Ridge (alpha=10):
  RMSE entrenamiento: 35,205
  RMSE prueba:        35,207


**Interpreta los resultados:** El modelo Ridge con alpha=10 es altamente recomendable porque:

Resolvió el problema de sobreajuste del modelo polinómico

*   Mantiene un error bajo (≈35,200 dólares de error típico)
*   Es robusto y generaliza bien






**Implementar Regresión Lasso**

Haz lo mismo, pero ahora con `Lasso(alpha=500, max_iter=10000)`. Lasso necesita un `alpha` más grande (porque la penalización L1 es diferente) y a veces más `max_iter` para converger.

In [14]:
# Crear el pipeline para Lasso
lasso_pipeline = Pipeline([
    ('poly_features', PolynomialFeatures(degree=5, include_bias=False)),
    ('scaler', StandardScaler()),
    ('regressor', Lasso(alpha=500, max_iter=10000))
])

In [15]:
# Entrenar el modelo
lasso_pipeline.fit(X_train, y_train)

y_train_pred_lasso = lasso_pipeline.predict(X_train)
y_test_pred_lasso = lasso_pipeline.predict(X_test)

rmse_train_lasso = np.sqrt(mean_squared_error(y_train, y_train_pred_lasso))
rmse_test_lasso = np.sqrt(mean_squared_error(y_test, y_test_pred_lasso))

In [16]:
# Calcular el error (RMSE) en entrenamiento y prueba y realizar un print de estos
print(f"Modelo Lasso (alpha=500):")
print(f"  RMSE entrenamiento: {rmse_train_lasso:,.0f}")
print(f"  RMSE prueba:        {rmse_test_lasso:,.0f}")

Modelo Lasso (alpha=500):
  RMSE entrenamiento: 35,353
  RMSE prueba:        35,448


**Interpreta los resultados:** Lasso ofrece una precisión casi idéntica a Ridge, pero con la enorme ventaja de que simplifica el modelo eliminando variables irrelevantes, para una firma inmobiliaria que necesita explicar por qué se predice cierto precio, Lasso puede ser más valioso a pesar de su mínima pérdida de precisión.

**Selección de Variables con Lasso**

Una de las grandes ventajas de Lasso es que puede eliminar variables. Vamos a ver cuántas de las características polinómicas que creamos (ej. `OverallQual^2`, `GrLivArea^5`, `OverallQual * GrLivArea^4`, etc.) fueron eliminadas.

In [17]:
# Extraer los coeficientes del modelo Lasso
coeficientes_lasso = lasso_pipeline.named_steps['regressor'].coef_

# Nombres de las características polinómicas
feature_names = lasso_pipeline.named_steps['poly_features'].get_feature_names_out(X.columns)

# Contar coeficientes exactamente cero (considerando tolerancia cercana a cero)
cero = np.isclose(coeficientes_lasso, 0)
num_ceros = np.sum(cero)
num_total = len(coeficientes_lasso)
num_conservadas = num_total - num_ceros
porcentaje_eliminadas = (num_ceros / num_total) * 100

print("\n--- Selección de variables con Lasso ---")
print(f"Número total de características polinómicas generadas: {num_total}")
print(f"Número de características eliminadas por Lasso (coef = 0): {num_ceros}")
print(f"Número de características CONSERVADAS: {num_conservadas}")
print(f"Porcentaje de variables eliminadas: {porcentaje_eliminadas:.1f}%")
print("\nCaracterísticas conservadas por Lasso (coeficiente != 0):")
for name, coef in zip(feature_names, coeficientes_lasso):
    if not np.isclose(coef, 0):
        print(f"  - {name}: {coef:.2f}")



--- Selección de variables con Lasso ---
Número total de características polinómicas generadas: 20
Número de características eliminadas por Lasso (coef = 0): 14
Número de características CONSERVADAS: 6
Porcentaje de variables eliminadas: 70.0%

Características conservadas por Lasso (coeficiente != 0):
  - OverallQual: 1963.77
  - OverallQual GrLivArea: 35444.98
  - OverallQual^2 GrLivArea: 10560.01
  - OverallQual^3 GrLivArea: 21555.10
  - OverallQual^5: 9492.34
  - GrLivArea^5: -20625.70


**Interpreta los resultados:** Aunque Ridge tiene un RMSE ligeramente mejor (diferencia de ~0.7%), Lasso ofrece un modelo mucho más interpretable y sencillo al eliminar el 70% de las variables, para un negocio que necesita transparencia y explicabilidad, esta ventaja compensa con creces la mínima pérdida de precisión.



### **Ejercicio 5: Conclusión y Recomendación para el Cliente**

**Resumir los Resultados**

Crea un `DataFrame` de pandas que compare el RMSE de entrenamiento y prueba de los tres modelos (Polinómico, Ridge, Lasso) y ordena los modelos según el RMSE de prueba de menor a mayor.

In [18]:
# Crear DataFrame de resultados
resultados = pd.DataFrame({
    'Modelo': ['Polinómico (grado 5)', 'Ridge (alpha=10)', 'Lasso (alpha=500)'],
    'RMSE_Entrenamiento': [rmse_train, rmse_train_ridge, rmse_train_lasso],
    'RMSE_Prueba': [rmse_test, rmse_test_ridge, rmse_test_lasso]
})

# Ordenar por RMSE de prueba (menor a mayor)
resultados_ordenados = resultados.sort_values('RMSE_Prueba')
print(resultados_ordenados)

                 Modelo  RMSE_Entrenamiento  RMSE_Prueba
1      Ridge (alpha=10)           35,204.89    35,206.63
2     Lasso (alpha=500)           35,352.92    35,447.71
0  Polinómico (grado 5)           34,152.35    41,101.60


**Pregunta Final:**

Basado en tu análisis, ¿qué modelo le recomendarías a la firma inmobiliaria? Tu respuesta debe incluir:

1.  Una recomendación clara del mejor modelo.
La diferencia entre Ridge y Lasso es de solo 241 dólares en el error típico de predicción. En el contexto de precios de vivienda (que van desde decenas de miles hasta cientos de miles de dólares), esta diferencia es prácticamente irrelevante.

Sin embargo, Lasso ofrece una ventaja enorme en interpretabilidad que Ridge no puede proporcionar.
2.  Una explicación en términos sencillos (para un gerente, no para un científico de datos) de por qué el modelo polinómico simple no era una buena idea, usando el concepto de "memorizar vs. generalizar".

Un ejemplo es un estudiante que estudia para un examen de matematicas, hay dos formas de estudiar:
Memorizar: Aprende de memoria las respuestas de 100 problemas específicos. Si el examen contiene exactamente esos mismos problemas, sacará 10/10. Pero si el examen tiene problemas nuevos, aunque sean del mismo tema, fracasará estrepitosamente.

Generalizar: Aprende los principios y fórmulas, quizás no resuelva perfectamente los problemas que ya vio, pero podrá resolver CUALQUIER problema nuevo que se le presente.

Nuestro modelo polinómico grado 5 "memorizó" las 70% de las casas usadas para entrenarlo, aprendió patrones demasiado específicos, ruido y casualidades particulares de esas casas, por eso:

Error en entrenamiento: 34,152 (muy bajo, parece buen estudiante)

Error en prueba: 41,102 (mucho más alto, fracasa con casas nuevas)

En cambio, Ridge y Lasso "generalizan": Su error apenas cambia entre entrenamiento y prueba, lo que significa que funcionarán bien con casas nuevas que la firma quiera valorar.


3.  Una descripción de la ventaja principal del modelo Lasso en cuanto a la interpretabilidad y la selección de las características más importantes.
El problema de la mayoría de los modelos de Machine Learning:

Son como una "caja negra": ingresas datos, sale una predicción, pero nadie sabe por qué, esto es inaceptable para decisiones de negocio importantes como valorar una propiedad.

La ventaja de Lasso:

Lasso no solo predice bien, sino que elimina automáticamente las variables que no sirven.

En nuestro caso:

Generamos 20 variables polinómicas (OverallQual, GrLivArea, OverallQual × GrLivArea, etc.)

Lasso eliminó 14 de ellas (70%) asignándoles coeficiente cero

Solo conservó las 6 realmente importantes

**Conclusión final para la firma inmobiliaria**

Recomiendo implementar el modelo Lasso, con un error típico de aproximadamente 35,000 dólares, su precisión es prácticamente idéntica a la del mejor modelo (Ridge), pero con una ventaja decisiva: podemos explicar a cualquier cliente o regulador POR QUÉ se predice un precio determinado. Lasso nos dice exactamente qué características de la vivienda importan y cuáles son ruido, en un negocio donde la confianza y la transparencia son clave, esto es invaluable.